# Compare the Production and destruction profiles for different turbulence models

In [1]:
import matplotlib.pyplot as plt
import numpy as np

from os import makedirs
from glob import glob
from pandas import read_csv
from os.path import join, exists

In [2]:
# validation @ Ma = 0.73
u_inf = 242.16629

# chord length
chord = 1

# use latex fonts
plt.style.use("default")
plt.rcParams.update({"text.usetex": True, "figure.dpi": 360})

# use these line styles
ls = ["-", "--", "-.", ":"]

In [3]:
# SALSA vs. exp. data
load_dir = join("/media", "janis", "Elements", "Janis", "2D_buffet_simulation", "URANS_2D_Ma0.73_Re3e6")
save_dir = join("..", "run", "plots", "URANS_validation", "URANS_blockMesh", "SALSA", "revised_new_mesh", "comparison_production_destruction_terms")
cases = ["URANS_SA_alpha3.5deg_blockMesh_newMesh", "URANS_SA_cc_alpha3.5deg_blockMesh_newMesh", "URANS_SALSA_alpha3.5deg_blockMesh_useRmod_useSmod_newMesh"]

legend = [r"$\mathrm{SA}$", r"$\mathrm{SA-CC}$", r"$\mathrm{SALSA}$"]

In [26]:
# load the line samples for the specified locations and times
locations = ["0.28", "0.45", "0.6", "0.75"]
loc = ["028", "045", "06", "075"]

# timings for mean / min / max cl for the unsteady cases
# SALSA: ["0.952", "0.916", "0.932"] (max, min, mean)
# SA-CC: ["0.935", "0.902", "0.918"] (max, min, mean)
write_time = "0.952"

# map the labels
if write_time == "0.932":
    label = "mean"
    wt_SA_cc = "0.918"
elif write_time == "0.952":
    label = "max"
    wt_SA_cc = "0.935"
else:
    label = "min"
    wt_SA_cc = "0.902"

In [20]:
# create plot directory
if not exists(save_dir):
    makedirs(save_dir)

In [27]:
# SA model constants
Cb1 = 0.1355
Cb2 = 0         # for SALSA
Cw2 = 0.3
Cw3 = 2
kappa = 0.41
sigma = 2/3
Cv1 = 7.1
Cv1_3 = Cv1**3

# common free stream quantities
rho0 = 0.957837
mu = 7.7319e-05
gamma = 1.4

# SA-CC
C5 = 3.5

def compute_cc(_rho, _nuTilda, _gamma, line):
    gradU2 = (line["gradU_xx"]**2 + line["gradU_xz"]**2 + line["gradU_zx"]**2 + line["gradU_zz"]**2)
    a2 = _gamma * line["p"].values / _rho
    return C5 * _rho * _nuTilda * gradU2 / a2

def compute_Sstar(line):
    Sxx = line["gradU_xx"]
    Sxz = 0.5*(line["gradU_xz"] + line["gradU_zx"])
    Szz = line["gradU_zz"]
    trace = Sxx + Szz

    Sdev_xx = Sxx - trace/3
    Sdev_yy = -trace/3
    Sdev_zz = Szz - trace/3
    Sdev_xz = Sxz

    # S^*
    return np.sqrt(2.0 * (Sdev_xx**2 + Sdev_yy**2 + Sdev_zz**2 + 2.0*Sdev_xz**2))

def compute_omega(line):
    dUxdz = line["gradU_xz"].values
    dUzdx = line["gradU_zx"].values

    return np.abs(dUxdz - dUzdx)

def compute_Stilde_SALSA(_rho, _nuTilda, S):
    chi = _nuTilda * _rho / mu
    fv1 = chi**3 / (chi**3 + Cv1_3)
    factor = 1.0 / np.maximum(chi, 1e-15) + fv1

    # Stilde = S^* (or Stilde) * (1/chi + fv1)
    return S * factor

def compute_cw1(_sqrtGamma, cb2):
    return (Cb1 * _sqrtGamma) / kappa**2 + (1 + cb2) / sigma

def compute_Edwards_mod(_rho, _nuTilda, _d, _Stilde):
    psi_ = np.sqrt(rho0 / _rho) * (_nuTilda / (kappa**2 * np.maximum(_d**2, 1e-15)))
    return 1.6 * np.tanh( 0.7 * (psi_ / _Stilde))

def compute_r(_nuTilda, _d, _Stilde):
        return np.minimum(_nuTilda / (kappa**2 * np.maximum(_d**2, 1e-15)) * (1 / _Stilde), 10.0)

def compute_fw(_rho, _nuTilda, _d, _Stilde, edwards: bool):
    if edwards:
        r_ = compute_Edwards_mod(_rho, _nuTilda, _d, _Stilde)
    else:
        r_ = compute_r(_nuTilda, _d, _Stilde)

    g_ = r_ * (1 + Cw2 * (r_**5 - 1))
    return g_ * ( (1 + Cw3**6) / (g_**6 + Cw3**6))**(1/6)

def compute_Stilde_SA(_rho, _nuTilda, _omega, _d):
    chi = _rho * _nuTilda / mu
    fv1 = chi**3 / (chi**3 + Cv1_3)
    fv2 = 1.0 - chi / (1.0 + chi * fv1)

    return _omega + fv2 * _nuTilda/(kappa**2*np.maximum(_d**2, 1e-15))

def compute_bl_thickness(_u, _z):
    # get idx of the maximum and crop everything after the max.
    max_idx = _u.argmax()
    u99 = 0.99 * _u[max_idx]

    # interpolate the z-coordinate at 0.99 * Ue
    _delta = np.interp(u99, _u[:max_idx], _z[:max_idx])
    return _delta

In [28]:
# initialize empty lists for results
production, destruction, compressibility, sqrtGamma, all_z, all_u = [], [], [], [], [], []

for c in range(len(cases)):
    # define the columns to load
    if "SALSA" in cases[c]:
        cols = [0, 2, 4, 7, 8, 9, 11, 12, 14, 15, 17, 20, 21, 23, 27, 29]
        names = ["z", "nuTilda", "p", "rho", "sqrtGamma", "Ux", "Uz", "Ux_mean", "Uz_mean", "U_xx", "U_xz", "U_zz", "gradU_xx", "gradU_xz", "gradU_zx", "gradU_zz"]
    else:
        cols = [0, 2, 3, 6, 7, 9, 10, 12, 13, 15, 18, 19, 21, 25, 27]
        names = ["z", "nuTilda", "p", "rho", "Ux", "Uz", "Ux_mean", "Uz_mean", "U_xx", "U_xz", "U_zz", "gradU_xx", "gradU_xz", "gradU_zx", "gradU_zz"]

    # make sure we compare the same states wrt shock position
    wt = wt_SA_cc if c == 1 else write_time

    # load the line samples
    try:
        files = [glob(join(load_dir, cases[c], "postProcessing", "sample_lines", wt, f"xc_{l}_*.csv"))[0] for l in loc]
    except IndexError:
        # for SA we only have the last dt, since it reaches a steady state anyway
        files = [glob(join(load_dir, cases[c], "postProcessing", "sample_lines", "1", f"xc_{l}_*.csv"))[0] for l in loc]

    lines = [read_csv(f, names=names, header=None, sep=",", skiprows=1, usecols=cols) for f in files]
    z, g, cc, p, des, u = [], [], [], [], [], []

    # loop over each sampling line and compute the production & destruction terms
    for s in range(len(locations)):
        # extract common variables
        rho = lines[s]["rho"].values
        nuTilda = lines[s]["nuTilda"].values

        # for wall distance omit the zero term, not sure how accurate this is, but shouldn't be much of an issue
        d = (lines[s]["z"].values - lines[s]["z"][0])

        # compute the compressibility correction for SA-CC
        if "SA_cc" in cases[c]:
            cc.append(compute_cc(rho, nuTilda, gamma, lines[s]))
        else:
            cc.append([])

        # compute Sstar or Stilde
        if "SALSA" in cases[c]:
            Sstar = compute_Sstar(lines[s])

            # if SA -> Stilde instead of SA, rest remains the same
            Stilde = compute_Stilde_SALSA(rho, nuTilda, Sstar)

            # compute the production
            g.append(lines[s]["sqrtGamma"])
            p.append(Cb1 * Stilde * nuTilda * rho * lines[s]["sqrtGamma"])
        else:
            omega = compute_omega(lines[s])
            Stilde = compute_Stilde_SA(rho, nuTilda, omega, d)
            p.append(Cb1 * Stilde * nuTilda * rho)
            g.append([])

        # compute destruction
        if "SALSA" in cases[c]:
            cw1 = compute_cw1(lines[s]["sqrtGamma"], Cb2)
            # compute fw
            fw = compute_fw(rho, nuTilda, d, Stilde, True)
        else:
            cw1 = compute_cw1(np.ones(Stilde.shape), 0.622)
            # compute fw
            fw = compute_fw(rho, nuTilda, d, Stilde, False)

        z.append(d)
        des.append((cw1 * fw) * nuTilda**2 / np.maximum(d**2, 1e-15) * rho)
        u.append(np.sqrt(lines[s]["Ux_mean"].values**2 + lines[s]["Uz_mean"].values**2))

    all_z.append(z)
    destruction.append(des)
    production.append(p)
    sqrtGamma.append(g)
    compressibility.append(cc)
    all_u.append(u)

In [ ]:
# plot the CC and sqrt gamm profiles to check if the effect is the same
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")
ax2 = [a.twiny() for a in ax]

for i in range(len(locations)):
    # compute approx. BL height
    delta99_SAcc = compute_bl_thickness(all_u[1][i], all_z[1][i])
    delta99_SALSA = compute_bl_thickness(all_u[2][i], all_z[2][i])

    if i == 0:
        ax2[i].plot(-compressibility[1][i], all_z[1][i] / delta99_SAcc, zorder=10, color="red", marker="none", ls="--",
                    label=r"$-c_5 \frac{\tilde{\nu}^2}{a^2}\frac{\partial u_i}{\partial x_j}\frac{\partial u_i}{\partial x_j}$")
        ax[i].plot(sqrtGamma[2][i], all_z[2][i] / delta99_SALSA, zorder=10, color="black", marker="none", ls="-", label=r"$\sqrt\Gamma$")
    else:
        ax2[i].plot(-compressibility[1][i], all_z[1][i] / delta99_SAcc, zorder=10, color="red", marker="none", ls="--")
        ax[i].plot(sqrtGamma[2][i], all_z[2][i] / delta99_SALSA, zorder=10, color="black", marker="none", ls="-")

    ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
    ax[i].minorticks_on()
    ax[i].tick_params(axis="both", which="minor", bottom=True)
    ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
    ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
    # ax[i].set_ylim(0, 0.06)
    ax[i].set_ylim(0, 2)
    ax[i].set_xlim(0.86, 1.14)
    # ax[i].set_xscale("log")
    ax[i].set_yscale("log")

    # second x-axis
    ax2[i].tick_params(axis="x", colors="red")
    # ax2[i].set_xlim(-1, 1)
    ax2[i].minorticks_on()
    ax2[i].spines["top"].set_color("red")
    ax2[i].tick_params(axis="x", which="minor", colors="red")

# ax[0].set_ylabel(r"$y / c$")
ax[0].set_ylabel(r"$y / \delta$")

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.24)
plt.savefig(join(save_dir, f"sqrtGamma_CC_profiles_with_ratio_log_cl_{label}_scaled_with_delta.png"))
plt.show()

In [ ]:
# plot the production & destruction terms
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")

for c in range(len(cases)):
    for i in range(len(locations)):
        # compute approx. BL height
        delta99 = compute_bl_thickness(all_u[c][i], all_z[c][i])

        if i == 0:
            ax[i].plot((production[c][i] - destruction[c][i]) / (production[c][i] + destruction[c][i]), all_z[c][i] / delta99, zorder=10, color="black",
                        marker="none", ls=ls[c], label=legend[c])
        else:
            ax[i].plot((production[c][i] - destruction[c][i]) / (production[c][i] + destruction[c][i]), all_z[c][i] / delta99, zorder=10, color="black",
                        marker="none", ls=ls[c])

        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="both", which="minor", bottom=True)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
        ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
        # ax[i].set_ylim(0, 0.06)
        ax[i].set_ylim(0, 1)
        ax[i].set_xlim(-1.05, 1.05)
        # ax[i].set_xscale("log")
        ax[i].set_yscale("log")

# ax[0].set_ylabel(r"$y / c$")
ax[0].set_ylabel(r"$y / \delta$")

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.2)
plt.savefig(join(save_dir, f"production_destruction_ratios_profiles_with_ratio_log_cl_{label}_scaled_with_delta.png"))
plt.show()

In [ ]:
# plot the production & destruction terms
fig, ax = plt.subplots(nrows= 1, ncols=4, figsize=(6, 3), sharey="row")
# ax2 = [a.twiny() for a in ax]

for c in range(len(cases)):
    for i in range(len(locations)):
        # compute approx. BL height
        delta99 = compute_bl_thickness(all_u[c][i], all_z[c][i])
        if i == 0:
            ax[i].plot(production[c][i], all_z[c][i] / delta99, zorder=10, color="black", marker="none", ls=ls[c], label=legend[c])
            ax[i].plot(destruction[c][i], all_z[c][i] / delta99, zorder=10, color="red", marker="none", ls=ls[c])
        else:
            ax[i].plot(production[c][i], all_z[c][i] / delta99, zorder=10, color="black",marker="none", ls=ls[c])
            ax[i].plot(destruction[c][i], all_z[c][i] / delta99, zorder=10, color="red",marker="none", ls=ls[c])

        ax[i].grid(visible=True, which="major", linestyle="-", alpha=0.35, color="black", axis="both")
        ax[i].minorticks_on()
        ax[i].tick_params(axis="both", which="minor", bottom=True)
        ax[i].grid(visible=True, which="minor", linestyle="--", alpha=0.25, color="black", axis="both")
        ax[i].set_title(fr"$ x / c = {float(locations[i]):.2f}$")
        # ax[i].set_ylim(0, 0.06)
        ax[i].set_ylim(0, 1)
        ax[i].set_xlim(0, 200)
        # ax[i].set_xscale("log")
        ax[i].set_yscale("log")

# ax[0].set_ylabel(r"$y / c$")
ax[0].set_ylabel(r"$y / \delta$")
ax[0].legend([r"$P_{\tilde{\nu}}$", r"$D_{\tilde{\nu}}$"])

fig.legend(ncol=4, loc="lower center")
fig.tight_layout()
fig.subplots_adjust(bottom=0.2)
plt.savefig(join(save_dir, f"production_destruction_profiles_with_ratio_log_cl_{label}_scaled_with_delta.png"))
plt.show()